# Approach A1: BiLSTM Dual Encoder from Scratch
## Khmer Legal Information Retrieval — Deep Learning Final Project

This notebook trains and benchmarks **Approach A1** on the official Cambodian Civil Code (2007) and Criminal Code (2009):
- **Architecture**: Bidirectional LSTM Dual Encoder with learnable word embeddings and mean pooling
- **Tokenization**: `khmer-nltk` dictionary/CRF word segmentation
- **Loss**: InfoNCE with in-batch negatives ($\tau = 0.05$)
- **Parameters**: ~1.97M trainable parameters (trained completely from scratch)
- **Optimizer**: Adam with learning rate search $\in \{0.0005, 0.001\}$ and hidden dimensions $\in \{128, 256\}$
- **Benchmarks**: Primary T-Q (200 human-verified questions) and Secondary T-T (148 article titles)

### Step 1: Check Environment
Verify Python and PyTorch environment.

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

### Step 2: Clone Repository & Install Dependencies

In [ ]:
!git clone https://github.com/mengchheanglong/khmer-legal-retrieval.git
%cd khmer-legal-retrieval
!pip install -q -r requirements.txt

### Step 3: Run Approach A1 Unit Tests

In [ ]:
!pytest tests/unit/test_a1_bilstm.py -v

### Step 4: Execute 4-Configuration Hyperparameter Grid Search
Grid: Learning Rate $\in \{0.0005, 0.001\} \times \text{Hidden Dimension} \in \{128, 256\}$.

In [ ]:
!python -m src.dl.experiments.tune_a1 --epochs 10 --batch-size 32 --patience 3

### Step 5: Display Hyperparameter Tuning Summary Table

In [ ]:
import pandas as pd
df_a1 = pd.read_csv('results/tuning/a1.csv')
print(df_a1.to_string(index=False))

### Step 6: Plot Training Dynamics
Plot per-epoch training loss and validation MRR@10 curves, illustrating the severe overfitting characteristic of from-scratch models on small corpora.

In [ ]:
import glob
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for f in sorted(glob.glob('results/logs/a1_*.csv')):
    name = f.split('/')[-1].replace('.csv', '').replace('a1_', '')
    df_run = pd.read_csv(f)
    ax1.plot(df_run['epoch'], df_run['train_loss'], marker='o', label=name)
    ax2.plot(df_run['epoch'], df_run['val_mrr10'], marker='s', label=name)

ax1.set_title('Training Loss (InfoNCE)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend()

ax2.set_title('Validation MRR@10')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MRR@10')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend()

plt.tight_layout()
plt.show()

### Step 7: View Final Benchmark Results on Full 1,976-Article Corpus

In [ ]:
import json
with open('results/metrics/a1_bilstm.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

print("=== Primary Benchmark (T-Q: 200 Questions) ===")
for metric, vals in results['tq_benchmark']['overall'].items():
    print(f"{metric:10s}: {vals['mean']:.4f} [{vals['ci_lower']:.4f}, {vals['ci_upper']:.4f}]")

print("\n=== Secondary Benchmark (T-T: 148 Titles) ===")
for metric, vals in results['tt_benchmark']['overall'].items():
    print(f"{metric:10s}: {vals['mean']:.4f} [{vals['ci_lower']:.4f}, {vals['ci_upper']:.4f}]")